[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# SQL Syntax &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, opens a
connection, and defines `JOINED`, the `FROM` clause the notebook used. Run it first. The tasks do not
depend on one another, and the last cell closes the connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


conn = sqlite3.connect(DATABASE)
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.commit()
JOINED = "FROM readings AS r JOIN stations AS s ON s.id = r.station_id"

print("built", DATABASE)


built scratch/stations.db


**1.** Svalbard's three coldest days in January.


In [2]:
for row in conn.execute(f"""
    SELECT date(r.hour) AS day, ROUND(AVG(r.celsius), 1) AS mean
    {JOINED}
    WHERE s.name = 'Svalbard' AND r.hour >= '2025-01-01' AND r.hour < '2025-02-01'
    GROUP BY date(r.hour)
    ORDER BY AVG(r.celsius), day
    LIMIT 3
"""):
    print(row)


('2025-01-19', -13.6)
('2025-01-15', -13.5)
('2025-01-20', -13.5)


A range of text chose January, and `date` made a group of every day. `ORDER BY` sorts by the mean
before rounding: 15 and 20 January both round to -13.5, and only the unrounded means say which was
colder.


**2.** Readings, missing readings and frost for every station.


In [3]:
for row in conn.execute("""
    SELECT s.name,
           COUNT(r.celsius) AS readings,
           COUNT(r.id) - COUNT(r.celsius) AS missing,
           ROUND(100.0 * AVG(r.celsius < 0), 1) AS frost_percent
    FROM stations AS s LEFT JOIN readings AS r ON r.station_id = s.id
    GROUP BY s.id
    ORDER BY s.name
"""):
    print(row)


('Bergen', 8760, 0, 13.9)
('Kirkenes', 0, 0, None)
('Oslo', 8760, 0, 21.1)
('Svalbard', 8736, 24, 67.2)
('Tromso', 8760, 0, 36.6)


The left join keeps Kirkenes, as one row whose columns from `readings` are all `NULL`. That row is
why the missing readings are `COUNT(r.id) - COUNT(r.celsius)`: `COUNT(*)` counts it, and would have
given Kirkenes one missing reading it never had. Its share of frost is `None`, the average of no
values, and `100.0` keeps every other share from being divided away.


**3.** Tromso's weekend readings in February.


In [4]:
weekend_readings = conn.execute(f"""
    SELECT COUNT(r.celsius)
    {JOINED}
    WHERE s.name = 'Tromso'
      AND r.hour >= '2025-02-01' AND r.hour < '2025-03-01'
      AND strftime('%w', r.hour) IN ('0', '6')
""").fetchone()[0]

print("Tromso's readings on February's weekends:", weekend_readings)


Tromso's readings on February's weekends: 192


February 2025 began on a Saturday, so it had four Saturdays and four Sundays, eight days of 24
readings. `strftime` returns text, so the days of the week are compared with `'0'` and `'6'`:
written as the numbers `0` and `6`, the same condition matches nothing, since text is never equal to
a number.


**4.** The hours between Svalbard's coldest reading and its warmest.


In [5]:
coldest = conn.execute(f"""
    SELECT r.hour, r.celsius {JOINED} WHERE s.name = 'Svalbard' ORDER BY r.celsius NULLS LAST, r.hour LIMIT 1
""").fetchone()
warmest = conn.execute(f"""
    SELECT r.hour, r.celsius {JOINED} WHERE s.name = 'Svalbard' ORDER BY r.celsius DESC, r.hour LIMIT 1
""").fetchone()
hours = conn.execute("SELECT ROUND((julianday(?) - julianday(?)) * 24, 1)", (warmest[0], coldest[0])).fetchone()[0]

print("coldest:", coldest)
print("warmest:", warmest)
print("hours between:", hours)


coldest: ('2025-01-12T03:00', -17.3)
warmest: ('2025-07-13T15:00', 8.3)
hours between: 4380.0


`NULLS LAST` keeps Svalbard's missing readings from sorting before its coldest one, and `r.hour`
would decide between two readings at the same temperature. The warmest reading came 4,380 hours after
the coldest, 182.5 days, and the difference is rounded because a Julian day is a real number.


**5.** Bergen's months with more than 100 hours below freezing.


In [6]:
for row in conn.execute(f"""
    SELECT strftime('%m', r.hour) AS month, COUNT(*) AS frost_hours
    {JOINED}
    WHERE s.name = 'Bergen' AND r.celsius < 0
    GROUP BY strftime('%m', r.hour)
    HAVING COUNT(*) > 100
    ORDER BY month
"""):
    print(row)


('01', 441)
('02', 323)
('12', 330)


`WHERE` kept only Bergen's readings below freezing before any grouping, so `COUNT(*)` counts frost
hours. `HAVING` then tested every month's count, and dropped March, with 86, and November, with 40.


**6.** A table made, changed, emptied of `NULL` and dropped.


In [7]:
conn.executescript("""
    CREATE TABLE monthly_coldest (station TEXT NOT NULL, month TEXT NOT NULL, coldest REAL);

    INSERT INTO monthly_coldest (station, month, coldest)
        SELECT s.name, strftime('%Y-%m', r.hour), MIN(r.celsius)
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE r.hour >= '2025-01-01' AND r.hour < '2026-01-01'
        GROUP BY s.id, strftime('%Y-%m', r.hour);

    UPDATE monthly_coldest SET coldest = NULL WHERE station = 'Oslo' AND month = '2025-12';
    DELETE FROM monthly_coldest WHERE coldest IS NULL;
""")
rows_left = conn.execute("SELECT COUNT(*) FROM monthly_coldest").fetchone()[0]
conn.execute("DROP TABLE monthly_coldest")

print("rows left:", rows_left)


rows left: 47


The `INSERT` added 48 rows, four stations by twelve months, since Kirkenes has no readings to join.
The `UPDATE` emptied Oslo's December and the `DELETE` removed it, which needs `IS NULL`:
`coldest = NULL` is never true, so it would have removed nothing.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [SQL Syntax](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/04-sql-syntax.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
